# The state of the AQLizer, on questions it has not been shown

Asking a question, watching it fail, adding an example that answers that question,
and reporting the pass is worthless. It measures whether the example was copied. Every
number in this notebook comes from questions chosen *after* the examples file was
finalised, aimed deliberately at parts of the graph no example query touches.

The file the AQLizer is given, `out/aql_examples.md`, is meant to teach the general
shape and the traps: which collections exist, what every field and edge label means
and which way it points, how a name from a question resolves, that a walk has to
cross typing, that units cannot be mixed, that a formula has no value, that a bind
parameter cannot run. What it must **not** do is contain the answer to the question
being asked. So the test is whether those general rules carry to a field, a label or
a collection the examples never query.

Each question below is asked in English. The AQL under it was written by a language
model at that moment. The `truth` beside it is a query written by hand in this
notebook, against the same graph, in a deliberately different shape.

There are two sets. The first fifteen were asked once, six caveats were then written
into the hand-written half of the examples file in response, and the second fifteen
were written afterwards against the changed file. Only the second set is a
measurement; the first is there to show nothing regressed. The caveats added prose
and tables only -- **no new example queries** -- so anything the second set gets right
is a rule carrying, not a query being copied.

In [1]:
import json, re, time
from collections import Counter

from sysml import config, nl

db = config.db()
E, D = config.ELEMENTS, config.DECLARATIONS
S, SIM = config.SOURCES, config.SIMILARITIES

def q(aql):
    """A hand-written query. Only ever used to check an answer, never to make one."""
    return list(db.aql.execute(aql, max_runtime=180))

NUMBER = re.compile(r"-?\d[\d,]*\.?\d*")

WORDS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
         "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12}

def numbers(text):
    """Every figure the answer states, including the ones written as words.

    A summary writes "contributed by six components" as readily as "by 6", and a
    check that only reads digits marks that wrong.
    """
    found = {float(m.replace(",", "")) for m in NUMBER.findall(text or "")}
    lowered = (text or "").lower()
    return found | {float(v) for w, v in WORDS.items() if re.search(rf"{w}", lowered)}

def near(found, wanted, tolerance=0.005):
    """The answer states this number, at any magnitude English writes it at."""
    for scale in (1, 1e3, 1e6, 1e9):
        target = wanted / scale
        if any(abs(f - target) <= max(1e-9, abs(target) * tolerance) for f in found):
            return True
    return False

def says(text, *words):
    lowered = (text or "").lower()
    return all(str(w).lower() in lowered for w in words)

print(f"{config.DB_NAME}: {db.collection(E).count():,} elements, "
      f"{db.collection(D).count():,} declaration edges, "
      f"{len(config.kinds())} kinds, {len(config.relation_types())} relation labels")

dronegraph: 3,715 elements, 11,539 declaration edges, 58 kinds, 53 relation labels


## What the examples actually demonstrate, and what they leave alone

Before choosing questions, this measures the surface. A field named in the field
table but never used inside an ```aql block has been *described*, not
*demonstrated* — and describing it is all the examples file is supposed to do.

In [2]:
built = config.AQL_EXAMPLES_BUILT.read_text(encoding="utf-8")
example_aql = "\n".join(re.findall(r"```aql(.*?)```", built, re.S))
print(f"{len(re.findall(r'```aql', built))} example queries, "
      f"{len(example_aql):,} characters of AQL\n")

fields = sorted({f for row in q(f"FOR e IN {E} LIMIT 600 RETURN ATTRIBUTES(e)")
                 for f in row if not f.startswith("_")})
labels = [r["l"] for r in q(f"""FOR d IN {D} COLLECT l = d.label WITH COUNT INTO n
                                 FILTER n > 3 RETURN {{l, n}}""")]

untouched_fields = [f for f in fields if f not in example_aql]
untouched_labels = [l for l in labels if f'"{l}"' not in example_aql]
untouched_colls = [c for c in config.ALL_COLLECTIONS if c not in example_aql]

for name, items in (("element fields never used in an example query", untouched_fields),
                    ("populated edge labels never traversed", untouched_labels),
                    ("collections never queried", untouched_colls)):
    print(f"{len(items)} {name}:")
    print("   " + ", ".join(items) + "\n")

35 example queries, 14,676 characters of AQL

19 element fields never used in an example query:
   abstract, annotations, anonymous, comments, conjugated, direction, end, identity, individual, modifiers, multiplicity, reference, references, source_column, specializes, states, typed_by, value_operator, visibility

14 populated edge labels never traversed:
   accepts, asserts, bindsTo, conjugates, connects, dependsOn, doAction, imports, performs, referencesFeature, requires, specializes, subject, triggeredBy

7 collections never queried:
   sysml_sources, sysml_domains, sysml_similarities, sysml_corpus_relations, sysml_Documents, sysml_Chunks, sysml_Communities



## The questions

Fifteen, each aimed at something in that list. None of them resembles an example
query; several are about fields that exist only as a row in the field table.

In [3]:
CASES = [
    ("abstract", "Which part definitions are abstract, and how many are there in each model?",
     f"""FOR e IN {E}
           FILTER e.abstract == true AND e.kind == "part" AND e.is_definition == true
           COLLECT module = e.module WITH COUNT INTO n RETURN {{module, n}}""",
     lambda text, truth: all(near(numbers(text), r["n"]) for r in truth),
     "a boolean field that appears in no example"),

    ("individual", "How many elements are marked as individuals?",
     f"""RETURN {{marked: LENGTH(FOR e IN {E} FILTER e.individual == true RETURN 1)}}""",
     lambda text, truth: near(numbers(text), truth[0]["marked"]),
     "a modifier the parser keeps and nothing demonstrates"),

    ("variation", "What variation points does the drone model declare, and what are the "
                  "options for each?",
     f"""FOR e IN {E} FILTER e.variation == true AND e.module == "DroneModelLogical"
           LET options = (FOR v, d IN 1..1 INBOUND e {D}
                            FILTER d.label == "variantOf" RETURN v.display)
           FILTER LENGTH(options) > 0
           RETURN {{variation: e.display, options}}""",
     lambda text, truth: all(says(text, o) for r in truth for o in r["options"]),
     "an inbound walk on a label no example traverses"),

    ("multiplicity", "How many parts are declared with an unbounded multiplicity?",
     f"""RETURN {{unbounded: LENGTH(FOR e IN {E}
           FILTER e.multiplicity != null AND e.multiplicity.text LIKE "%*%" RETURN 1)}}""",
     lambda text, truth: near(numbers(text), truth[0]["unbounded"]),
     "a nested field with its own shape, never shown"),

    ("conjugated", "Which ports are conjugated, and where is each declared?",
     f"""FOR e IN {E} FILTER e.conjugated == true
           RETURN {{element: e.display, at: CONCAT(e.source_file, ":", e.source_line)}}""",
     lambda text, truth: sum(1 for r in truth
                             if says(text, r["element"].split("_")[-1])) >= len(truth) // 2,
     "a SysML idea with both a field and an edge label, neither demonstrated"),

    ("direction", "Across the whole corpus, how many declarations carry a direction, "
                   "broken down by which direction it is?",
     f"""FOR e IN {E} FILTER e.direction != null
           COLLECT direction = e.direction WITH COUNT INTO n
           SORT n DESC RETURN {{direction, n}}""",
     lambda text, truth: near(numbers(text), truth[0]["n"])
     and near(numbers(text), truth[1]["n"]),
     "two counts off one field the examples only list"),

    ("annotations", "Which metadata annotations are used in the corpus, and how often?",
     f"""FOR e IN {E} FILTER e.annotations != null
           FOR a IN e.annotations
             COLLECT name = a.name WITH COUNT INTO n
             SORT n DESC LIMIT 2 RETURN {{annotation: name, n}}""",
     lambda text, truth: all(says(text, r["annotation"]) for r in truth)
     and near(numbers(text), truth[0]["n"]),
     "an array of objects on the element, never touched"),

    ("imports", "Which namespace imports the most others, and how many does it import?",
     f"""FOR e IN {E}
           LET n = LENGTH(FOR v, d IN 1..1 OUTBOUND e {D}
                            FILTER d.label == "imports" RETURN 1)
           FILTER n > 0 SORT n DESC LIMIT 1
           RETURN {{namespace: e.display, imports: n}}""",
     lambda text, truth: near(numbers(text), truth[0]["imports"])
     and says(text, truth[0]["namespace"].split("_")[-1]),
     "a ranking over an untouched label"),

    ("performs", "Which part performs the most actions, and how many?",
     f"""FOR e IN {E}
           LET n = LENGTH(FOR v, d IN 1..1 OUTBOUND e {D}
                            FILTER d.label == "performs" RETURN 1)
           FILTER n > 0 SORT n DESC LIMIT 1
           RETURN {{part: e.display, actions: n}}""",
     lambda text, truth: near(numbers(text), truth[0]["actions"])
     and says(text, truth[0]["part"].split("_")[-1]),
     "behaviour attached to structure, on an untouched label"),

    ("enumerations", "What enumeration definitions are there, and what are the literals of "
                     "each?",
     f"""FOR e IN {E} FILTER e.kind == "enumeration" AND e.is_definition == true
           LET literals = (FOR v, d IN 1..1 OUTBOUND e {D}
                             FILTER d.label == "owns" RETURN v.name)
           FILTER LENGTH(literals) > 0
           RETURN {{enumeration: e.display, literals}}""",
     lambda text, truth: sum(1 for r in truth for l in r["literals"]
                             if says(text, l)) >= 3,
     "a kind the examples never filter on"),

    ("snapshots", "How many snapshots does each model declare?",
     f"""FOR e IN {E} FILTER e.kind == "snapshot"
           COLLECT module = e.module WITH COUNT INTO n
           SORT n DESC RETURN {{module, n}}""",
     lambda text, truth: near(numbers(text), truth[0]["n"]),
     "an occurrence kind, never filtered on"),

    ("specialized", "Which definition is specialised by the most other definitions, and by "
                    "how many?",
     f"""FOR e IN {E}
           LET n = LENGTH(FOR v, d IN 1..1 INBOUND e {D}
                            FILTER d.label == "specializes" RETURN 1)
           FILTER n > 0 SORT n DESC LIMIT 1
           RETURN {{definition: e.display, specialised_by: n}}""",
     lambda text, truth: near(numbers(text), truth[0]["specialised_by"])
     and says(text, truth[0]["definition"]),
     "inbound on the one label every example only mentions"),

    ("largest-file", "Which source file is the largest, by character count, and how many "
                     "lines does it have?",
     f"""FOR s IN {S} SORT s.characters DESC LIMIT 1
           RETURN {{file: s.filename, characters: s.characters, lines: s.lines}}""",
     lambda text, truth: near(numbers(text), truth[0]["characters"])
     and says(text, truth[0]["file"].split("/")[-1]),
     "a Layer 2 collection no example query names"),

    ("similar-files", "Which two source files are the most similar to each other, and what "
                      "is the score?",
     f"""FOR s IN {SIM} SORT s.similarity_score DESC LIMIT 1
           RETURN {{a: DOCUMENT(s._from).filename, b: DOCUMENT(s._to).filename,
                    score: s.similarity_score}}""",
     lambda text, truth: says(text, truth[0]["a"].split("/")[-1])
     and says(text, truth[0]["b"].split("/")[-1]),
     "an edge collection with endpoints in another collection, never shown"),

    ("subject-of", "In the drone model, what is each requirement the subject of? Name the "
                   "requirement and what it is about.",
     f"""FOR d IN {D} FILTER d.label == "subject"
           LET holder = DOCUMENT(d._from), about = DOCUMENT(d._to)
           FILTER holder.module == "DroneModelLogical"
           RETURN {{holder: holder.display, subject: about.display}}""",
     lambda text, truth: all(says(text, r["subject"]) for r in truth),
     "a label added after the examples were written"),
]
print(f"{len(CASES)} held-out questions")

15 held-out questions


Ask them. This first set has been asked before, and the examples file has been
changed since, so treat what follows as a regression check: the questions are known
and the caveats were written knowing they existed.

In [4]:
def ask_all(cases, with_examples):
    out = []
    for name, question, truth_aql, check, why in cases:
        truth = q(truth_aql)
        started = time.time()
        try:
            answer = nl.instance().ask(question, with_examples=with_examples)
        except Exception as exc:
            answer = nl.Answer(question, "", error=repr(exc))
        try:
            ok = bool(truth) and bool(check(answer.answer or "", truth))
        except Exception:
            ok = False
        out.append({"case": name, "question": question, "why": why, "truth": truth,
                    "answer": answer.answer, "aql": answer.aql, "error": answer.error,
                    "rows": len(answer.rows or []), "note": answer.note, "ok": ok,
                    "seconds": time.time() - started})
    return out

SEEN = ask_all(CASES, True)
print(f"the first fifteen, re-asked: "
      f"{sum(1 for r in SEEN if r['ok'])}/{len(SEEN)} with the examples file")

LLM provider initialized successfully.


Connecting to ArangoDB at http://localhost:8529 (timeout=300s)


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


...

Successfully injected AQL reference into fix prompt.


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


the first fifteen, re-asked: 12/15 with the examples file


### Every question, with the query it produced

The AQL is what the model wrote. `truth` is the hand-written query from the cell
above. A `check` line disagreeing means the English answer does not state what the
graph holds.

In [5]:
for r in SEEN:
    print("=" * 100)
    print(f"{'PASS' if r['ok'] else 'FAIL'}  {r['case']}  --  {r['why']}")
    print(f"\nQ  {r['question']}")
    if r["aql"]:
        print("\nAQL")
        for line in r["aql"].strip().splitlines():
            print(f"   {line}")
    if r["error"]:
        print(f"\n!! {r['error']}")
    print(f"\nA  {' '.join((r['answer'] or '').split())[:600]}")
    print(f"\ntruth  {json.dumps(r['truth'], default=str)[:340]}")
    if r["note"]:
        print(f"note   {r['note']}")
    print(f"({r['seconds']:.0f}s, {r['rows']} rows returned)\n")

PASS  abstract  --  a boolean field that appears in no example

Q  Which part definitions are abstract, and how many are there in each model?

AQL
   WITH sysml_elements
   FOR e IN sysml_elements
     FILTER e.kind == "part" AND e.is_definition == true AND e.abstract == true
     COLLECT module = e.module WITH COUNT INTO count
     RETURN { module, abstract_part_definitions: count }

A  The query identified abstract part definitions in two models. The "apollo-11-sysml-v2" model contains 14 abstract part definitions, while the "DroneModelLogical" model has 4 abstract part definitions.

truth  [{"module": "apollo-11-sysml-v2", "n": 14}, {"module": "DroneModelLogical", "n": 4}]
(9s, 2 rows returned)

PASS  individual  --  a modifier the parser keeps and nothing demonstrates

Q  How many elements are marked as individuals?

AQL
   WITH sysml_elements
   FOR e IN sysml_elements
     FILTER e.individual == true
     COLLECT WITH COUNT INTO count
   RETURN count

A  There are 139 elements ma

## A second set, written after the caveats

The fifteen above have now been seen, so anything changed in response to them cannot
honestly be measured on them again — the run above is a regression check, nothing
more. These fifteen were written against the examples file as it stands, and aim at a
different part of the surface: the sparse fields the sampled schema leaves out
entirely, the two Layer 1 and Layer 2 collections, the field-versus-label
distinction, and the counts that are already on the row.

What changed in the examples file between the two sets is prose only — six caveats
about how this pipeline writes a row, and **no new example queries**. That matters:
if the second set does better, it is because a rule carried, not because a query got
copied.

In [6]:
COMPOSITION_LABELS = '["owns", "typedBy", "subsets", "redefines"]'

FRESH = [
    ("anonymous", "How many declarations in the corpus have no name written in the source?",
     f"""RETURN {{unnamed: LENGTH(FOR e IN {E} FILTER e.anonymous == true RETURN 1)}}""",
     lambda text, truth: near(numbers(text), truth[0]["unnamed"]),
     "a boolean the schema shows and nothing demonstrates"),

    ("import-count", "How many import statements does each model contain?",
     f"""FOR e IN {E} FILTER e.kind == "import"
           COLLECT module = e.module WITH COUNT INTO n SORT n DESC RETURN {{module, n}}""",
     lambda text, truth: all(near(numbers(text), r["n"]) for r in truth),
     "elements that no owns edge reaches"),

    ("external-types", "How many declarations name a type that is not declared anywhere in "
                       "the corpus?",
     f"""FOR e IN {E} FILTER LENGTH(e.typed_by) > 0
           LET edges = LENGTH(FOR v, d IN 1..1 OUTBOUND e {D}
                                FILTER d.label == "typedBy" RETURN 1)
           FILTER edges < LENGTH(e.typed_by)
           COLLECT WITH COUNT INTO n RETURN {{unresolved: n}}""",
     lambda text, truth: near(numbers(text), truth[0]["unresolved"]),
     "the field-versus-label split, which decides what an absence means"),

    ("most-children", "Which declaration has the most other declarations written directly "
                      "inside it, and how many?",
     f"""FOR e IN {E} SORT e.children DESC LIMIT 1
           RETURN {{element: e.display, children: e.children}}""",
     lambda text, truth: near(numbers(text), truth[0]["children"])
     and says(text, truth[0]["element"]),
     "a count already on the row, needing no traversal"),

    ("constraints", "What do the constraint expressions in the corpus say? Give a few.",
     f"""FOR e IN {E} FILTER e.expression != null LIMIT 4
           RETURN {{element: e.display, says: e.expression.text}}""",
     lambda text, truth: sum(1 for r in truth
                             if says(text, r["says"].split()[0])) >= 2,
     "a sparse field the sampled schema omits entirely"),

    ("rationale", "How many declarations carry a rationale?",
     f"""RETURN {{with_rationale: LENGTH(FOR e IN {E} FILTER e.rationale != null RETURN 1)}}""",
     lambda text, truth: near(numbers(text), truth[0]["with_rationale"]),
     "a field written only when an annotation carries text"),

    ("short-names", "How many declarations have a short name?",
     f"""RETURN {{with_short_name: LENGTH(FOR e IN {E}
           FILTER e.short_name != null RETURN 1)}}""",
     lambda text, truth: near(numbers(text), truth[0]["with_short_name"]),
     "the alternate name form, never counted in an example"),

    ("doc-search", "Which declarations have documentation that mentions contamination?",
     f"""FOR e IN {E} FILTER e.doc != null AND CONTAINS(LOWER(e.doc), "contamination")
           RETURN {{element: e.display, at: CONCAT(e.source_file, ":", e.source_line)}}""",
     lambda text, truth: sum(1 for r in truth
                             if says(text, r["element"].split("_")[-1])) >= 2,
     "a text search over a field no example reads"),

    ("both-forms", "How many element kinds appear both as definitions and as usages?",
     f"""LET pairs = (FOR e IN {E}
           COLLECT kind = e.kind, isdef = e.is_definition WITH COUNT INTO n
           RETURN {{kind, isdef}})
         FOR row IN pairs COLLECT kind = row.kind INTO rows
           FILTER LENGTH(rows) == 2 COLLECT WITH COUNT INTO n
           RETURN {{kinds_with_both: n}}""",
     lambda text, truth: near(numbers(text), truth[0]["kinds_with_both"]),
     "is_definition as a grouping rather than a filter"),

    ("module-files", "How many source files does each module contain?",
     f"""FOR m IN {config.MODULES} RETURN {{module: m.name, files: LENGTH(m.files)}}""",
     lambda text, truth: all(near(numbers(text), r["files"]) for r in truth),
     "the Layer 1 collection, and the length of a list on it"),

    ("domains", "How many document clusters did the corpus build produce?",
     f"""FOR d IN {config.DOMAINS} RETURN {{cluster: d._key}}""",
     lambda text, truth: near(numbers(text), len(truth)),
     "a Layer 2 collection no example query names"),

    ("recursive-imports", "How many imports are recursive, reaching into nested namespaces?",
     f"""FOR d IN {D} FILTER d.recursive != null
           COLLECT recursive = d.recursive WITH COUNT INTO n RETURN {{recursive, n}}""",
     lambda text, truth: near(numbers(text),
                              next(r["n"] for r in truth if r["recursive"])),
     "a sparse field on an edge, absent from the schema"),

    ("connected-through", "Which parts are connected to each other? Give a few pairs.",
     f"""FOR d IN {D} FILTER d.through != null LIMIT 3
           RETURN {{a: DOCUMENT(d._from).display, b: DOCUMENT(d._to).display}}""",
     lambda text, truth: sum(1 for r in truth
                             if says(text, r["a"]) or says(text, r["b"])) >= 2,
     "a sparse edge field that names a third element"),

    ("visibility", "How many declarations are marked private?",
     f"""FOR e IN {E} FILTER e.visibility != null
           COLLECT visibility = e.visibility WITH COUNT INTO n RETURN {{visibility, n}}""",
     lambda text, truth: near(numbers(text), truth[0]["n"]),
     "a field the schema types as NoneType because the sample lacked it"),

    ("power-rollup", "What is the total power load of the technical components package, and "
                     "how many components contribute to it?",
     f"""LET root = FIRST(FOR e IN {E}
           FILTER e.display == "TECHNICALCOMPONENTSPACKAGE" RETURN e)
         FOR v, edge, p IN 1..8 OUTBOUND root {D}
           OPTIONS {{bfs: true, uniqueVertices: "global"}}
           FILTER p.edges[*].label ALL IN {COMPOSITION_LABELS}
           FILTER IS_NUMBER(v.attributes.powerLoad.value)
           COLLECT unit = v.attributes.powerLoad.unit
           AGGREGATE total = SUM(v.attributes.powerLoad.value), n = COUNT()
           RETURN {{unit, total, contributors: n}}""",
     lambda text, truth: near(numbers(text), truth[0]["total"])
     and near(numbers(text), truth[0]["contributors"]),
     "a rollup on an attribute and a subject used nowhere before"),
]

FRESH_RESULTS = ask_all(FRESH, True)
print(f"the second set, never seen: "
      f"{sum(1 for r in FRESH_RESULTS if r['ok'])}/{len(FRESH_RESULTS)} with the examples file")

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


...

Successfully injected AQL reference into fix prompt.


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


the second set, never seen: 10/15 with the examples file


### Every question in the second set, with the query it produced

In [7]:
for r in FRESH_RESULTS:
    print("=" * 100)
    print(f"{'PASS' if r['ok'] else 'FAIL'}  {r['case']}  --  {r['why']}")
    print(f"\nQ  {r['question']}")
    if r["aql"]:
        print("\nAQL")
        for line in r["aql"].strip().splitlines():
            print(f"   {line}")
    if r["error"]:
        print(f"\n!! {r['error']}")
    print(f"\nA  {' '.join((r['answer'] or '').split())[:520]}")
    print(f"\ntruth  {json.dumps(r['truth'], default=str)[:300]}")
    if r["note"]:
        print(f"note   {r['note']}")
    print(f"({r['seconds']:.0f}s, {r['rows']} rows returned)\n")

PASS  anonymous  --  a boolean the schema shows and nothing demonstrates

Q  How many declarations in the corpus have no name written in the source?

AQL
   WITH sysml_elements
   FOR e IN sysml_elements
     FILTER e.anonymous == true
     COLLECT WITH COUNT INTO count
   RETURN count

A  There are 1,142 declarations in the corpus that have no name written in the source.

truth  [{"unnamed": 1142}]
(2s, 1 rows returned)

PASS  import-count  --  elements that no owns edge reaches

Q  How many import statements does each model contain?

AQL
   WITH sysml_elements
   FOR e IN sysml_elements
     FILTER e.kind == "import"
     COLLECT module = e.module WITH COUNT INTO importCount
     RETURN { module, importCount }

A  The query results indicate the number of import statements in each model. The "apollo-11-sysml-v2" model contains 159 import statements, the "Drone_BaseArchitecture" model has 1 import statement, and the "DroneModelLogical" model includes 7 import statements.

truth  [{"mod

## The second set with nothing but the schema

The service as deployed passes no `aql_examples` at all. This is the baseline.

In [8]:
BARE = ask_all(FRESH, False)
by_case = {r["case"]: r for r in BARE}

print(f"{'case':<19}{'examples':<11}{'schema only':<13}what it needs")
print("-" * 108)
for r in FRESH_RESULTS:
    other = by_case.get(r["case"], {})
    print(f"{r['case']:<19}"
          f"{'PASS' if r['ok'] else 'FAIL':<11}"
          f"{'PASS' if other.get('ok') else 'FAIL':<13}{r['why']}")
print("-" * 108)
print(f"{'':<19}{sum(1 for r in FRESH_RESULTS if r['ok'])}/{len(FRESH_RESULTS):<9}"
      f"{sum(1 for r in BARE if r['ok'])}/{len(BARE)}")
print()
print(f"and the first, already-seen set, for regression: "
      f"{sum(1 for r in SEEN if r['ok'])}/{len(SEEN)}")

Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


...

Successfully injected AQL reference into fix prompt.


Loaded AQL reference documentation from C:\Users\Christian\Desktop\arango\natural-language-service\txt2aql\prompts\aql_ref.md (104940 chars)


Successfully injected AQL reference into generation prompt.


Successfully injected AQL reference into fix prompt.


case               examples   schema only  what it needs
------------------------------------------------------------------------------------------------------------
anonymous          PASS       FAIL         a boolean the schema shows and nothing demonstrates
import-count       PASS       FAIL         elements that no owns edge reaches
external-types     PASS       FAIL         the field-versus-label split, which decides what an absence means
most-children      PASS       FAIL         a count already on the row, needing no traversal
constraints        FAIL       FAIL         a sparse field the sampled schema omits entirely
rationale          PASS       FAIL         a field written only when an annotation carries text
short-names        PASS       FAIL         the alternate name form, never counted in an example
doc-search         PASS       PASS         a text search over a field no example reads
both-forms         FAIL       PASS         is_definition as a grouping rather than a filt

## Where the second set missed

In [9]:
for r in FRESH_RESULTS:
    if r["ok"]:
        continue
    print("=" * 100)
    print(f"{r['case']}  --  {r['why']}")
    print(f"Q      {r['question']}")
    print(f"truth  {json.dumps(r['truth'], default=str)[:280]}")
    print(f"said   {' '.join((r['answer'] or r['error'] or '').split())[:380]}")
    print(f"rows   {r['rows']}")
    if r["aql"]:
        print("aql")
        for line in r["aql"].strip().splitlines():
            print(f"   {line}")
    print()
if all(r["ok"] for r in FRESH_RESULTS):
    print("nothing missed")

constraints  --  a sparse field the sampled schema omits entirely
Q      What do the constraint expressions in the corpus say? Give a few.
truth  [{"element": "CONSTRAINT@32", "says": "drone.totalMass <= 750"}, {"element": "CONSTRAINT@41", "says": "battery.capacity >= 6000"}, {"element": "SYSTEMPOWERANALYSIS_CONSTRAINT@28", "says": "powerMargin > 0[W]"}, {"element": "CONSTRAINT@149", "says": "ascentMargin > 0['m\u22c5s\u20
said   The constraint expressions in the corpus reveal several conditions related to the Apollo 11 mission. For instance, the "SystemPowerAnalysis" requires that the power margin be greater than 0 watts. The "Apollo11MissionDeltaVBudgetAnalysis" specifies that various margins, such as ascent and landing, must be greater than 0 meters per second. Additionally, the "crewIngress" and "li
rows   5
aql
   WITH sysml_elements
   FOR c IN sysml_elements
     FILTER c.kind == "constraint"
     FILTER c.expression != null
     SORT c.source_file, c.source_line
     LIMIT 5
  

## What four runs of this actually establish

Scores from every run of these two sets, as the examples file changed underneath
them:

| | run 1 | run 2 | run 3 | run 4 |
| --- | --- | --- | --- | --- |
| first set, with examples | 11/15 | 11/15 | 13/15 | 12/15 |
| second set, with examples | — | 11/15 | 12/15 | 10/15 |
| second set, schema only | — | 3/15 | — | 4/15 |

Two things follow, and only two.

**The examples file is worth roughly three times the schema.** 10–13 out of 15 with
it against 3–4 without, on questions aimed at fields and collections no example query
touches. That gap is far larger than the movement between runs, so it is the one
number here that means something.

**Nothing smaller than that gap can be measured one run at a time.** The swing on an
unchanged question is ±2 or 3 of 15, which is the same size as any single caveat's
plausible effect. Run 4 scored lower than run 3 while containing strictly more
guidance, and the three questions that flipped — a count on a Layer 2 collection, a
recursive-import filter, a connection walk — each went wrong in a different way from
the run before, none of them related to what had been added. Attributing that to the
change would be reading noise.

So the caveats added in this session cannot be credited from these runs, and the
honest claim is narrower: two of them are mechanically observable, because they are
not prose. `provenance_gap` reads the generated query, and when it finds
`attributes.<sparse field>` or a literal element key it names the fault and asks
again — the `note` lines above are those interventions firing. The rest is prose, and
prose is exactly what this sample size cannot resolve.

Measuring a single caveat would need each question asked several times per file
version, which multiplies the cost of a run by that factor. That is the price of a
number worth quoting, and it is worth knowing before anyone quotes one.

## Retrieval, for the questions that have no query

Nothing above asked what anything is *for*. That is not a gap in the AQL side: Layer
2 holds what the syntax states, and intent is not stated syntactically. The GraphRAG
retrievers answer from Layer 3 and the source text, with citations that resolve to a
file. Neither of these questions is in any example either.

In [10]:
async def rag(question, scope="local"):
    """Awaited rather than called: the kernel is already in a loop, so
    `nl.graphrag`'s `asyncio.run` cannot be used from a cell."""
    started = time.time()
    try:
        answer = await nl.retriever().ask_async(question, scope)
    except Exception as exc:
        print(f"!! {exc!r}")
        return nl.Answer(question, "", error=repr(exc))
    answer.show()
    print(f"({time.time() - started:.0f}s, {scope})")
    return answer

answer = await rag("What is the purpose of the conjugated ports on the command module, "
                  "and what do they connect to?")
answer.evidence(500)

RetrievalService initialized with chat_api_provider: openai, embedding_api_provider: openai


Processing local query: What is the purpose of the conjugated ports on the...


CACHE DISABLED (use_cache=False) - Skipping cache check


Creating LocalRetriever with provider: openai


Vector index for embedding already exists


...

LOCAL: model response length=1684


Successfully processed LOCAL query


Cache check: use_cache=False, is_dict=True, from_cache=None, stream=False, query_type=LOCAL


CACHE DISABLED (use_cache=False) - Skipping cache write


Q  What is the purpose of the conjugated ports on the command module, and what do they connect to?

retrieved  33 documents, 75 edges, 92,826 chars of context

cited (8, first 6)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Technical/TechnicalPortsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Technical/TechnicalPortsPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 4, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 5, "source": "models/DroneModelLogical.sysml"}
   {"cite": 6, "source": "models/DroneModelLogical.sysml"}

A  ## Purpose of the Conjugated Ports on the Command Module

The context provided gives some insight into the technical system of the Apollo spacecraft, specifically regarding ports and interfaces. However, it does not mention "conjugated ports" explicitly with respect to the Command Module. Therefore, inferring their purp

In [11]:
await rag("What engineering concerns do these models cover, taken together?",
          scope="global")

Processing global query: What engineering concerns do these models cover, t...


CACHE DISABLED (use_cache=False) - Skipping cache check


Creating ArangoGlobalRetriever instance


ArangoGlobalRetriever created successfully


Communities collection has embeddings: True


...

Successfully processed GLOBAL query


Cache check: use_cache=False, is_dict=True, from_cache=None, stream=False, query_type=GLOBAL


CACHE DISABLED (use_cache=False) - Skipping cache write


Q  What engineering concerns do these models cover, taken together?

retrieved  84 community reports -> 28 points

A  # Engineering Concerns Covered by the Models

## Lunar Landing and Mission Safety

The **LunarLanderSoftLandingRequirement** specifies vital attributes for a successful lunar landing, such as velocities and zone deviations, to ensure precision in navigation, guidance, and control during mission execution.

## Propulsion Systems and Configuration

Numerous models focus on the propulsion systems for aerospace vehicles. These are apparent in the **TECHNICALREQUIREMENTSPACKAGE**, which details engine configurations and capabilities essential for mission success.

## Command Module and Interface Systems

The **COMMANDMODULE** plays a crucial role by interfacing with key components like the Launch Escape System. Its critical attributes—including oxygen levels, heatshield temperature, gForce, and cabin pressure—ensure both the safety and success of the mission.

## Crew Activi

Answer(question='What engineering concerns do these models cover, taken together?', answer="# Engineering Concerns Covered by the Models\n\n## Lunar Landing and Mission Safety\n\nThe **LunarLanderSoftLandingRequirement** specifies vital attributes for a successful lunar landing, such as velocities and zone deviations, to ensure precision in navigation, guidance, and control during mission execution.\n\n## Propulsion Systems and Configuration\n\nNumerous models focus on the propulsion systems for aerospace vehicles. These are apparent in the **TECHNICALREQUIREMENTSPACKAGE**, which details engine configurations and capabilities essential for mission success.\n\n## Command Module and Interface Systems\n\nThe **COMMANDMODULE** plays a crucial role by interfacing with key components like the Launch Escape System. Its critical attributes—including oxygen levels, heatshield temperature, gForce, and cabin pressure—ensure both the safety and success of the mission.\n\n## Crew Activity and Well-

## Reading this

`out/aql_examples.md` is two halves: a hand-written one that may not mention any
corpus, and a generated one written from a survey of whatever is loaded. Neither is
allowed to contain the answer to a question, and the questions above were chosen
against the finished file rather than the file being adjusted until they passed.

To re-run this against a rebuilt graph:

```
python build.py                      # corpus -> extract -> load -> examples -> probe
python -m sysml.pipeline.corpus.check     # every relation label has a producer
python -m sysml.pipeline.examples.probe   # eight corpus-agnostic questions, scored
python -m sysml.nl "..."                  # one question, from a shell
```

If a question here fails, the honest fix is a general rule or a caveat in the
hand-written half — something true of any corpus this pipeline builds. Adding an
example that answers this particular question would make the number go up and mean
nothing.